In [ ]:
#@title Cell 27.1 - Notebook overview
from IPython.display import display, Markdown

display(Markdown(r"""
# Notebook 27: Model C Pathogen Similarity and Prediction Support

## Purpose

Notebook 27 will determine whether Model C prediction error increases when a
pathogen is less similar to the pathogens used for model training.

For each pathogen held out during Notebook 26 validation, it will:

1. identify the pathogens used to train that outer-fold model;
2. calculate the held-out pathogen's Model C similarity to those training
   pathogens;
3. record its highest similarity, representing its nearest training pathogen;
4. combine this similarity with its held-out MIC prediction errors;
5. summarise prediction error across different similarity ranges;
6. test whether prediction error increases as similarity decreases;
7. define the lowest nearest-training similarity evaluated during pathogen-out
   validation;
8. use this value as the empirical boundary of the validated prediction domain;
9. classify future pathogens as inside or outside the evaluated similarity
   range; and
10. save the support information required by the final prediction notebook.

## Model C specification

Notebook 26 selected

\[
K_P^{(C)}
=
0.4K_{\mathrm{seq,locus}}
+
0.6K_P^{(3B)}.
\]

The final model uses 256 pathogen coordinates, 26 antibiotic coordinates and
Ridge \(\alpha=1\).

## Model boundary

Notebook 27 will not refit Model C or change its selected settings.

Notebook 25 and Notebook 26 files will be treated as read-only inputs. For each
outer fold, a held-out pathogen will be compared only with pathogens in that
fold's training group.

Notebook 27 will not fill the sparse MIC matrix and will not predict MICs for a
new pathogen. Those operations belong to later notebooks.

## Expected notebook length

Notebook 27 contains **10 cells**.
"""))

print(
    "Transition: The next cell will import the required packages "
    "and define the Notebook 27 settings."
)


In [ ]:
# =============================================================================
# Cell 27.2
# =============================================================================

#@title Cell 27.2 - Import packages and define notebook settings
# This cell imports the required packages, mounts Google Drive and defines the
# fixed cohort size, fold count, calculation settings and output directories.

import gc
import hashlib
import json
import math
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from google.colab import drive
from IPython.display import display
from scipy.stats import spearmanr


drive.mount(
    "/content/drive",
    force_remount=False,
)


EXPECTED_MODEL_C_PATHOGENS = 9058
EXPECTED_OUTER_FOLDS = 5
SIMILARITY_ROW_BLOCK_SIZE = 128
BOOTSTRAP_REPLICATES = 2000
RANDOM_SEED = 42

SIMILARITY_PERCENTILES = [
    0,
    1,
    5,
    10,
    25,
    50,
    75,
    90,
    95,
    99,
    100,
]


MYDRIVE_DIRECTORY = Path(
    "/content/drive/MyDrive"
)

PROJECT_DIRECTORY = (
    MYDRIVE_DIRECTORY
    / "Model3_MIC_Project"
)

NOTEBOOK25_DIRECTORY = (
    PROJECT_DIRECTORY
    / "notebook25"
)

NOTEBOOK26_DIRECTORY = (
    PROJECT_DIRECTORY
    / "notebook26"
)

NOTEBOOK27_DIRECTORY = (
    PROJECT_DIRECTORY
    / "notebook27"
)

NOTEBOOK27_RESULT_DIRECTORY = (
    NOTEBOOK27_DIRECTORY
    / "results"
)

FOLD_CHECKPOINT_DIRECTORY = (
    NOTEBOOK27_DIRECTORY
    / "fold_checkpoints"
)

WORK_DIRECTORY = Path(
    "/content/notebook27_work"
)

INPUT_DIRECTORY = (
    WORK_DIRECTORY
    / "inputs"
)

NOTEBOOK25_INPUT_DIRECTORY = (
    INPUT_DIRECTORY
    / "notebook25"
)

NOTEBOOK26_INPUT_DIRECTORY = (
    INPUT_DIRECTORY
    / "notebook26"
)

for directory in [
    PROJECT_DIRECTORY,
    NOTEBOOK27_DIRECTORY,
    NOTEBOOK27_RESULT_DIRECTORY,
    FOLD_CHECKPOINT_DIRECTORY,
    WORK_DIRECTORY,
    INPUT_DIRECTORY,
    NOTEBOOK25_INPUT_DIRECTORY,
    NOTEBOOK26_INPUT_DIRECTORY,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


NOTEBOOK25_ARCHIVE_FILENAME = (
    "25_model_c_pathogen_kernel_candidate_components.zip"
)

NOTEBOOK26_ARCHIVE_FILENAME = (
    "26_model_c_nested_pathogen_out_and_reference_model_outputs.zip"
)


def file_sha256(file_path):
    digest = hashlib.sha256()

    with open(
        file_path,
        "rb",
    ) as input_file:
        for block in iter(
            lambda: input_file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


settings_summary = pd.DataFrame(
    [
        {
            "setting": "Model C pathogens",
            "value": EXPECTED_MODEL_C_PATHOGENS,
        },
        {
            "setting": "Outer pathogen folds",
            "value": EXPECTED_OUTER_FOLDS,
        },
        {
            "setting": "Similarity row-block size",
            "value": SIMILARITY_ROW_BLOCK_SIZE,
        },
        {
            "setting": "Bootstrap replicates",
            "value": BOOTSTRAP_REPLICATES,
        },
        {
            "setting": "Notebook 27 output directory",
            "value": str(NOTEBOOK27_DIRECTORY),
        },
    ]
)

display(settings_summary)

print(
    "Notebook 27 packages, directories and fixed settings "
    "were defined successfully."
)

print(
    "\nTransition: Cell 27.3 will locate, validate and "
    "extract the required Notebook 25 and Notebook 26 inputs."
)


In [ ]:
# =============================================================================
# Cell 27.3
# =============================================================================

#@title Cell 27.3 - Locate, validate and extract the required archives
# This cell finds the Notebook 25 kernel-component archive and the Notebook 26
# validation archive, checks their ZIP integrity and extracts only required files.


def locate_archive(
    authoritative_filename,
    preferred_paths,
):
    authoritative_path = Path(
        authoritative_filename
    )

    candidate_paths = []

    for preferred_path in preferred_paths:
        preferred_path = Path(
            preferred_path
        )

        if preferred_path.exists():
            candidate_paths.append(
                preferred_path.resolve()
            )

    if not candidate_paths:
        filename_stem = authoritative_path.stem

        candidate_paths = [
            path.resolve()
            for path in PROJECT_DIRECTORY.rglob(
                f"{filename_stem}*.zip"
            )
            if path.name.startswith(
                filename_stem
            )
        ]

    candidate_paths = sorted(
        set(candidate_paths),
        key=lambda path: (
            len(path.parts),
            str(path),
        ),
    )

    if not candidate_paths:
        raise FileNotFoundError(
            f"{authoritative_filename} was not found. "
            f"Copy it under {PROJECT_DIRECTORY} and rerun "
            "Cell 27.3."
        )

    if len(candidate_paths) > 1:
        candidate_hashes = {
            file_sha256(path)
            for path in candidate_paths
        }

        if len(candidate_hashes) > 1:
            raise ValueError(
                f"Multiple different copies of "
                f"{authoritative_filename} were found: "
                f"{candidate_paths}. Retain one authoritative "
                "copy and rerun Cell 27.3."
            )

    return candidate_paths[0]


def archive_member_for_basename(
    archive,
    required_basename,
):
    matches = [
        member
        for member in archive.namelist()
        if Path(member).name
        == required_basename
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected one archive member named "
            f"{required_basename}, but found {matches}."
        )

    return matches[0]


def validate_and_extract_members(
    archive_path,
    required_basenames,
    destination_directory,
):
    extracted_paths = {}

    with zipfile.ZipFile(
        archive_path,
        "r",
    ) as archive:
        damaged_member = archive.testzip()

        if damaged_member is not None:
            raise ValueError(
                f"{archive_path.name} contains a damaged "
                f"member: {damaged_member}"
            )

        for required_basename in required_basenames:
            archive_member = archive_member_for_basename(
                archive,
                required_basename,
            )

            output_path = (
                destination_directory
                / required_basename
            )

            partial_path = Path(
                str(output_path)
                + ".partial"
            )

            partial_path.unlink(
                missing_ok=True
            )

            with archive.open(
                archive_member,
                "r",
            ) as source_file:
                with open(
                    partial_path,
                    "wb",
                ) as destination_file:
                    shutil.copyfileobj(
                        source_file,
                        destination_file,
                        length=1024 * 1024,
                    )

            partial_path.replace(
                output_path
            )

            extracted_paths[
                required_basename
            ] = output_path

    return extracted_paths


notebook25_archive_path = locate_archive(
    NOTEBOOK25_ARCHIVE_FILENAME,
    [
        NOTEBOOK25_DIRECTORY
        / NOTEBOOK25_ARCHIVE_FILENAME,
        PROJECT_DIRECTORY
        / NOTEBOOK25_ARCHIVE_FILENAME,
    ],
)

notebook26_archive_path = locate_archive(
    NOTEBOOK26_ARCHIVE_FILENAME,
    [
        NOTEBOOK26_DIRECTORY
        / NOTEBOOK26_ARCHIVE_FILENAME,
        PROJECT_DIRECTORY
        / NOTEBOOK26_ARCHIVE_FILENAME,
    ],
)


notebook25_required_files = [
    "25_model3b_pathogen_kernel_subset.npy",
    "25_sequence_kernel_locus_weighted.npy",
    "25_model_c_pathogen_order.csv",
    "25_output_manifest.json",
]

notebook26_required_files = [
    "26_biosample_outer_fold_assignments.csv",
    "26_nested_pathogen_out_predictions.csv.gz",
    "26_nested_overall_performance.csv",
    "26_final_model_c_configuration.json",
    "26_output_manifest.json",
]


notebook25_input_paths = validate_and_extract_members(
    notebook25_archive_path,
    notebook25_required_files,
    NOTEBOOK25_INPUT_DIRECTORY,
)

notebook26_input_paths = validate_and_extract_members(
    notebook26_archive_path,
    notebook26_required_files,
    NOTEBOOK26_INPUT_DIRECTORY,
)


input_summary = pd.DataFrame(
    [
        {
            "input": "Notebook 25 kernel components",
            "archive": notebook25_archive_path.name,
            "required_files": len(
                notebook25_required_files
            ),
            "validation_status": "passed",
        },
        {
            "input": "Notebook 26 validation results",
            "archive": notebook26_archive_path.name,
            "required_files": len(
                notebook26_required_files
            ),
            "validation_status": "passed",
        },
    ]
)

display(input_summary)

print(f"Notebook 25 archive: {notebook25_archive_path}")
print(f"Notebook 26 archive: {notebook26_archive_path}")

print(
    "\nTransition: Cell 27.4 will load and validate the "
    "Model C kernel components, pathogen folds and predictions."
)


In [ ]:
# =============================================================================
# Cell 27.4
# =============================================================================

#@title Cell 27.4 - Load and validate the Model C inputs
# This cell verifies the final Model C settings, kernel checksums, pathogen
# order, outer-fold assignments and held-out prediction records.

with open(
    notebook25_input_paths[
        "25_output_manifest.json"
    ],
    "r",
    encoding="utf-8",
) as manifest_file:
    notebook25_manifest = json.load(
        manifest_file
    )

with open(
    notebook26_input_paths[
        "26_final_model_c_configuration.json"
    ],
    "r",
    encoding="utf-8",
) as configuration_file:
    model_c_configuration = json.load(
        configuration_file
    )

with open(
    notebook26_input_paths[
        "26_output_manifest.json"
    ],
    "r",
    encoding="utf-8",
) as manifest_file:
    notebook26_manifest = json.load(
        manifest_file
    )

if notebook26_manifest.get(
    "validation_status"
) != "passed":
    raise ValueError(
        "Notebook 26 did not report passed validation."
    )

if model_c_configuration.get(
    "validation_status"
) != "passed":
    raise ValueError(
        "The final Model C configuration did not pass validation."
    )


notebook25_file_records = {
    record["file_name"]: record
    for record in notebook25_manifest[
        "files"
    ]
}

notebook26_file_records = {
    record["file_name"]: record
    for record in notebook26_manifest[
        "files"
    ]
}

for input_paths, file_records, manifest_name in [
    (
        notebook25_input_paths,
        notebook25_file_records,
        "25_output_manifest.json",
    ),
    (
        notebook26_input_paths,
        notebook26_file_records,
        "26_output_manifest.json",
    ),
]:
    for input_name, input_path in input_paths.items():
        if input_name == manifest_name:
            continue

        expected_record = file_records.get(
            input_name
        )

        if expected_record is None:
            raise ValueError(
                f"The source manifest does not contain {input_name}."
            )

        if file_sha256(
            input_path
        ) != expected_record[
            "sha256"
        ]:
            raise ValueError(
                f"The checksum failed for {input_name}."
            )

SELECTED_SEQUENCE_KERNEL = str(
    model_c_configuration[
        "selected_sequence_kernel"
    ]
)

SELECTED_RHO = float(
    model_c_configuration[
        "selected_rho"
    ]
)

SELECTED_PATHOGEN_DIMENSION = int(
    model_c_configuration[
        "selected_pathogen_dimension"
    ]
)

SELECTED_RIDGE_ALPHA = float(
    model_c_configuration[
        "selected_ridge_alpha"
    ]
)

if SELECTED_SEQUENCE_KERNEL != "locus-weighted":
    raise ValueError(
        "Notebook 27 requires the locus-weighted kernel "
        "selected by Notebook 26."
    )

if not 0.0 <= SELECTED_RHO <= 1.0:
    raise ValueError(
        "The selected rho value is outside [0, 1]."
    )


model_c_pathogen_order = pd.read_csv(
    notebook25_input_paths[
        "25_model_c_pathogen_order.csv"
    ]
).sort_values(
    "model_c_row_index"
).reset_index(drop=True)

fold_assignments = pd.read_csv(
    notebook26_input_paths[
        "26_biosample_outer_fold_assignments.csv"
    ]
)

nested_predictions = pd.read_csv(
    notebook26_input_paths[
        "26_nested_pathogen_out_predictions.csv.gz"
    ]
)

nested_overall_performance = pd.read_csv(
    notebook26_input_paths[
        "26_nested_overall_performance.csv"
    ]
)


required_order_columns = {
    "model_c_row_index",
    "biosample",
    "assembly_accession",
}

required_fold_columns = {
    "biosample",
    "outer_fold",
}

required_prediction_columns = {
    "interaction_row",
    "biosample",
    "antibiotic",
    "observed_log2_mic",
    "predicted_log2_mic",
    "outer_fold",
    "model",
}

if not required_order_columns.issubset(
    model_c_pathogen_order.columns
):
    raise ValueError(
        "The Model C pathogen order is missing required columns."
    )

if not required_fold_columns.issubset(
    fold_assignments.columns
):
    raise ValueError(
        "The outer-fold assignment table is missing required columns."
    )

if not required_prediction_columns.issubset(
    nested_predictions.columns
):
    raise ValueError(
        "The nested prediction table is missing required columns."
    )

if len(model_c_pathogen_order) != EXPECTED_MODEL_C_PATHOGENS:
    raise ValueError(
        "The Model C pathogen order has the wrong number of rows."
    )

if not np.array_equal(
    model_c_pathogen_order[
        "model_c_row_index"
    ].to_numpy(dtype=np.int64),
    np.arange(
        EXPECTED_MODEL_C_PATHOGENS,
        dtype=np.int64,
    ),
):
    raise ValueError(
        "The Model C pathogen rows are not complete and consecutive."
    )

if model_c_pathogen_order[
    "biosample"
].duplicated().any():
    raise ValueError(
        "Duplicate BioSamples were detected in the pathogen order."
    )

if (
    len(fold_assignments)
    != EXPECTED_MODEL_C_PATHOGENS
    or fold_assignments[
        "biosample"
    ].duplicated().any()
):
    raise ValueError(
        "The outer-fold assignment does not contain one row "
        "per Model C pathogen."
    )

observed_fold_numbers = sorted(
    fold_assignments[
        "outer_fold"
    ].astype(int).unique()
)

if observed_fold_numbers != list(
    range(1, EXPECTED_OUTER_FOLDS + 1)
):
    raise ValueError(
        "The expected five outer folds were not found."
    )


pathogen_fold_table = model_c_pathogen_order.merge(
    fold_assignments,
    on="biosample",
    how="left",
    validate="one_to_one",
)

if pathogen_fold_table[
    "outer_fold"
].isna().any():
    raise ValueError(
        "At least one Model C pathogen has no outer-fold assignment."
    )

pathogen_fold_table[
    "outer_fold"
] = pathogen_fold_table[
    "outer_fold"
].astype(int)


model_prediction_counts = nested_predictions[
    "model"
].value_counts()

EXPECTED_MODEL_C_INTERACTIONS = int(
    notebook26_manifest[
        "observed_interactions"
    ]
)

for model_name in [
    "Model C",
    "Model 3B baseline",
]:
    if int(
        model_prediction_counts.get(
            model_name,
            0,
        )
    ) != EXPECTED_MODEL_C_INTERACTIONS:
        raise ValueError(
            f"{model_name} does not contain one held-out "
            "prediction for every MIC observation."
        )

if not np.isfinite(
    nested_predictions[
        [
            "observed_log2_mic",
            "predicted_log2_mic",
        ]
    ].to_numpy(dtype=np.float64)
).all():
    raise ValueError(
        "The nested prediction table contains invalid values."
    )


kernel_component_paths = {
    "model3b": notebook25_input_paths[
        "25_model3b_pathogen_kernel_subset.npy"
    ],
    "locus": notebook25_input_paths[
        "25_sequence_kernel_locus_weighted.npy"
    ],
}

kernel_components = {}

for component_id, component_path in (
    kernel_component_paths.items()
):
    expected_file_record = (
        notebook25_file_records.get(
            component_path.name
        )
    )

    if expected_file_record is None:
        raise ValueError(
            f"The Notebook 25 manifest does not contain "
            f"{component_path.name}."
        )

    kernel = np.load(
        component_path,
        mmap_mode="r",
    )

    if kernel.shape != (
        EXPECTED_MODEL_C_PATHOGENS,
        EXPECTED_MODEL_C_PATHOGENS,
    ):
        raise ValueError(
            f"{component_path.name} has invalid dimensions."
        )

    maximum_diagonal_difference = float(
        np.max(
            np.abs(
                np.asarray(
                    np.diagonal(kernel),
                    dtype=np.float64,
                )
                - 1.0
            )
        )
    )

    if maximum_diagonal_difference > 1e-6:
        raise ValueError(
            f"{component_path.name} has an invalid diagonal."
        )

    kernel_components[
        component_id
    ] = kernel


input_validation_summary = pd.DataFrame(
    [
        {
            "metric": "Model C pathogens",
            "value": len(pathogen_fold_table),
        },
        {
            "metric": "Outer folds",
            "value": len(observed_fold_numbers),
        },
        {
            "metric": "Observed MIC interactions",
            "value": EXPECTED_MODEL_C_INTERACTIONS,
        },
        {
            "metric": "Selected sequence kernel",
            "value": SELECTED_SEQUENCE_KERNEL,
        },
        {
            "metric": "Selected rho",
            "value": SELECTED_RHO,
        },
        {
            "metric": "Selected pathogen coordinates",
            "value": SELECTED_PATHOGEN_DIMENSION,
        },
        {
            "metric": "Selected Ridge alpha",
            "value": SELECTED_RIDGE_ALPHA,
        },
        {
            "metric": "Input validation status",
            "value": "passed",
        },
    ]
)

display(input_validation_summary)

print(
    "Model C inputs, pathogen order, folds and held-out "
    "predictions were validated successfully."
)

print(
    "\nTransition: Cell 27.5 will calculate each held-out "
    "pathogen's nearest training-pathogen similarity."
)


In [ ]:
# =============================================================================
# Cell 27.5
# =============================================================================

#@title Cell 27.5 - Calculate nearest training-pathogen similarity
# This cell calculates the final Model C similarity between every held-out
# pathogen and the training pathogens from the same outer fold. Restartable
# checkpoints are saved separately for all five outer folds.


def fold_checkpoint_paths(
    fold_number,
):
    prefix = (
        f"27_outer_fold_{fold_number:02d}"
    )

    return {
        "similarity": (
            FOLD_CHECKPOINT_DIRECTORY
            / f"{prefix}_nearest_similarity.csv"
        ),
        "completion": (
            FOLD_CHECKPOINT_DIRECTORY
            / f"{prefix}_complete.json"
        ),
    }


def fold_checkpoint_is_valid(
    fold_number,
):
    paths = fold_checkpoint_paths(
        fold_number
    )

    if (
        not paths["similarity"].exists()
        or not paths["completion"].exists()
    ):
        return False

    try:
        with open(
            paths["completion"],
            "r",
            encoding="utf-8",
        ) as completion_file:
            completion = json.load(
                completion_file
            )

        expected_rows = int(
            (
                pathogen_fold_table[
                    "outer_fold"
                ]
                == fold_number
            ).sum()
        )

        return (
            completion.get("outer_fold")
            == fold_number
            and completion.get("pathogens")
            == expected_rows
            and completion.get("selected_rho")
            == SELECTED_RHO
            and completion.get(
                "selected_sequence_kernel"
            )
            == SELECTED_SEQUENCE_KERNEL
            and completion.get("file_sha256")
            == file_sha256(
                paths["similarity"]
            )
            and completion.get(
                "validation_status"
            )
            == "passed"
        )

    except Exception:
        return False


def write_csv_atomically(
    table,
    output_path,
):
    partial_path = Path(
        str(output_path)
        + ".partial"
    )

    partial_path.unlink(
        missing_ok=True
    )

    table.to_csv(
        partial_path,
        index=False,
    )

    partial_path.replace(
        output_path
    )


for fold_number in range(
    1,
    EXPECTED_OUTER_FOLDS + 1,
):
    checkpoint_paths = fold_checkpoint_paths(
        fold_number
    )

    if fold_checkpoint_is_valid(
        fold_number
    ):
        print(
            f"Reusing completed outer fold {fold_number}."
        )
        continue

    held_out_table = pathogen_fold_table[
        pathogen_fold_table[
            "outer_fold"
        ]
        == fold_number
    ].sort_values(
        "model_c_row_index"
    ).reset_index(drop=True)

    training_table = pathogen_fold_table[
        pathogen_fold_table[
            "outer_fold"
        ]
        != fold_number
    ].sort_values(
        "model_c_row_index"
    ).reset_index(drop=True)

    held_out_rows = held_out_table[
        "model_c_row_index"
    ].to_numpy(dtype=np.int64)

    training_rows = training_table[
        "model_c_row_index"
    ].to_numpy(dtype=np.int64)

    if set(held_out_rows).intersection(
        set(training_rows)
    ):
        raise ValueError(
            f"Outer fold {fold_number} contains pathogen-row leakage."
        )

    nearest_similarity_values = np.empty(
        len(held_out_rows),
        dtype=np.float32,
    )

    nearest_training_positions = np.empty(
        len(held_out_rows),
        dtype=np.int64,
    )

    print(
        f"Calculating outer fold {fold_number}: "
        f"{len(held_out_rows):,} held-out pathogens "
        f"against {len(training_rows):,} training pathogens."
    )

    for block_start in range(
        0,
        len(held_out_rows),
        SIMILARITY_ROW_BLOCK_SIZE,
    ):
        block_stop = min(
            block_start
            + SIMILARITY_ROW_BLOCK_SIZE,
            len(held_out_rows),
        )

        block_rows = held_out_rows[
            block_start:block_stop
        ]

        model3b_block = np.asarray(
            kernel_components[
                "model3b"
            ][
                block_rows,
                :,
            ][
                :,
                training_rows,
            ],
            dtype=np.float32,
        )

        locus_block = np.asarray(
            kernel_components[
                "locus"
            ][
                block_rows,
                :,
            ][
                :,
                training_rows,
            ],
            dtype=np.float32,
        )

        combined_block = (
            (1.0 - SELECTED_RHO)
            * model3b_block
            + SELECTED_RHO
            * locus_block
        )

        if not np.isfinite(
            combined_block
        ).all():
            raise ValueError(
                f"Outer fold {fold_number} produced invalid "
                "similarity values."
            )

        block_nearest_positions = np.argmax(
            combined_block,
            axis=1,
        )

        nearest_training_positions[
            block_start:block_stop
        ] = block_nearest_positions

        nearest_similarity_values[
            block_start:block_stop
        ] = combined_block[
            np.arange(
                block_stop - block_start
            ),
            block_nearest_positions,
        ]

        del model3b_block
        del locus_block
        del combined_block
        gc.collect()

    nearest_training_rows = training_rows[
        nearest_training_positions
    ]

    training_lookup = training_table.set_index(
        "model_c_row_index"
    )

    nearest_training_records = training_lookup.loc[
        nearest_training_rows
    ].reset_index()

    fold_similarity = pd.DataFrame(
        {
            "model_c_row_index": held_out_rows,
            "biosample": held_out_table[
                "biosample"
            ].to_numpy(),
            "assembly_accession": held_out_table[
                "assembly_accession"
            ].to_numpy(),
            "outer_fold": fold_number,
            "nearest_training_row":
                nearest_training_rows,
            "nearest_training_biosample":
                nearest_training_records[
                    "biosample"
                ].to_numpy(),
            "nearest_training_assembly_accession":
                nearest_training_records[
                    "assembly_accession"
                ].to_numpy(),
            "nearest_training_outer_fold":
                nearest_training_records[
                    "outer_fold"
                ].to_numpy(dtype=np.int64),
            "nearest_training_similarity":
                nearest_similarity_values,
        }
    )

    if (
        fold_similarity[
            "nearest_training_outer_fold"
        ]
        == fold_number
    ).any():
        raise ValueError(
            f"Outer fold {fold_number} selected a held-out "
            "pathogen as a training neighbour."
        )

    if not fold_similarity[
        "nearest_training_similarity"
    ].between(
        -1e-6,
        1.0 + 1e-6,
    ).all():
        raise ValueError(
            f"Outer fold {fold_number} contains a similarity "
            "outside [0, 1]."
        )

    write_csv_atomically(
        fold_similarity,
        checkpoint_paths["similarity"],
    )

    completion = {
        "outer_fold": fold_number,
        "pathogens": len(fold_similarity),
        "selected_sequence_kernel":
            SELECTED_SEQUENCE_KERNEL,
        "selected_rho": SELECTED_RHO,
        "file_sha256": file_sha256(
            checkpoint_paths["similarity"]
        ),
        "validation_status": "passed",
    }

    partial_completion_path = Path(
        str(checkpoint_paths["completion"])
        + ".partial"
    )

    with open(
        partial_completion_path,
        "w",
        encoding="utf-8",
    ) as completion_file:
        json.dump(
            completion,
            completion_file,
            indent=2,
        )

    partial_completion_path.replace(
        checkpoint_paths["completion"]
    )

    print(
        f"Outer fold {fold_number} saved and validated."
    )


if not all(
    fold_checkpoint_is_valid(
        fold_number
    )
    for fold_number in range(
        1,
        EXPECTED_OUTER_FOLDS + 1,
    )
):
    raise ValueError(
        "At least one nearest-similarity fold checkpoint failed."
    )


nearest_similarity = pd.concat(
    [
        pd.read_csv(
            fold_checkpoint_paths(
                fold_number
            )["similarity"]
        )
        for fold_number in range(
            1,
            EXPECTED_OUTER_FOLDS + 1,
        )
    ],
    ignore_index=True,
).sort_values(
    "model_c_row_index"
).reset_index(drop=True)

if (
    len(nearest_similarity)
    != EXPECTED_MODEL_C_PATHOGENS
    or nearest_similarity[
        "biosample"
    ].duplicated().any()
):
    raise ValueError(
        "The combined nearest-similarity table does not contain "
        "one row per Model C pathogen."
    )

NEAREST_SIMILARITY_PATH = (
    NOTEBOOK27_RESULT_DIRECTORY
    / "27_pathogen_nearest_training_similarity.csv"
)

nearest_similarity.to_csv(
    NEAREST_SIMILARITY_PATH,
    index=False,
)

nearest_similarity_summary = pd.DataFrame(
    [
        {
            "metric": "Pathogens evaluated",
            "value": len(nearest_similarity),
        },
        {
            "metric": "Minimum nearest-training similarity",
            "value": nearest_similarity[
                "nearest_training_similarity"
            ].min(),
        },
        {
            "metric": "Median nearest-training similarity",
            "value": nearest_similarity[
                "nearest_training_similarity"
            ].median(),
        },
        {
            "metric": "Maximum nearest-training similarity",
            "value": nearest_similarity[
                "nearest_training_similarity"
            ].max(),
        },
        {
            "metric": "Training-neighbour fold leakage",
            "value": int(
                (
                    nearest_similarity[
                        "outer_fold"
                    ]
                    == nearest_similarity[
                        "nearest_training_outer_fold"
                    ]
                ).sum()
            ),
        },
    ]
)

display(nearest_similarity_summary)

print(f"Saved: {NEAREST_SIMILARITY_PATH}")

print(
    "\nTransition: Cell 27.6 will combine nearest-training "
    "similarity with the held-out MIC prediction errors."
)


In [ ]:
# =============================================================================
# Cell 27.6
# =============================================================================

#@title Cell 27.6 - Combine pathogen similarity with held-out prediction errors
# This cell joins each pathogen's nearest training similarity to its Notebook 26
# held-out predictions and calculates MIC-level and pathogen-level errors.

predictions_with_support = nested_predictions.merge(
    nearest_similarity[
        [
            "biosample",
            "outer_fold",
            "model_c_row_index",
            "nearest_training_row",
            "nearest_training_biosample",
            "nearest_training_similarity",
        ]
    ],
    on=[
        "biosample",
        "outer_fold",
    ],
    how="left",
    validate="many_to_one",
)

if predictions_with_support[
    "nearest_training_similarity"
].isna().any():
    raise ValueError(
        "At least one held-out prediction has no nearest-training "
        "similarity value."
    )

predictions_with_support[
    "prediction_error"
] = (
    predictions_with_support[
        "predicted_log2_mic"
    ]
    - predictions_with_support[
        "observed_log2_mic"
    ]
)

predictions_with_support[
    "absolute_error"
] = predictions_with_support[
    "prediction_error"
].abs()

predictions_with_support[
    "squared_error"
] = np.square(
    predictions_with_support[
        "prediction_error"
    ]
)

if not np.isfinite(
    predictions_with_support[
        [
            "nearest_training_similarity",
            "prediction_error",
            "absolute_error",
            "squared_error",
        ]
    ].to_numpy(dtype=np.float64)
).all():
    raise ValueError(
        "The similarity-linked prediction table contains "
        "invalid numerical values."
    )


model_c_predictions = predictions_with_support[
    predictions_with_support[
        "model"
    ]
    == "Model C"
].copy()

if len(model_c_predictions) != (
    EXPECTED_MODEL_C_INTERACTIONS
):
    raise ValueError(
        "The Model C prediction table has the wrong number "
        "of MIC observations."
    )


pathogen_error_summary = (
    model_c_predictions.groupby(
        [
            "model_c_row_index",
            "biosample",
            "outer_fold",
            "nearest_training_row",
            "nearest_training_biosample",
            "nearest_training_similarity",
        ],
        as_index=False,
    )
    .agg(
        observed_mic_values=(
            "interaction_row",
            "size",
        ),
        pathogen_mae=(
            "absolute_error",
            "mean",
        ),
        pathogen_mean_squared_error=(
            "squared_error",
            "mean",
        ),
        mean_prediction_error=(
            "prediction_error",
            "mean",
        ),
    )
)

pathogen_error_summary[
    "pathogen_rmse"
] = np.sqrt(
    pathogen_error_summary[
        "pathogen_mean_squared_error"
    ]
)

pathogen_error_summary = pathogen_error_summary.drop(
    columns=[
        "pathogen_mean_squared_error"
    ]
)

if (
    len(pathogen_error_summary)
    != EXPECTED_MODEL_C_PATHOGENS
    or pathogen_error_summary[
        "biosample"
    ].duplicated().any()
):
    raise ValueError(
        "The pathogen-level error table does not contain "
        "one row per Model C pathogen."
    )


PREDICTIONS_WITH_SUPPORT_PATH = (
    NOTEBOOK27_RESULT_DIRECTORY
    / "27_nested_predictions_with_similarity.csv.gz"
)

PATHOGEN_ERROR_SUMMARY_PATH = (
    NOTEBOOK27_RESULT_DIRECTORY
    / "27_model_c_pathogen_error_and_similarity.csv"
)

predictions_with_support.to_csv(
    PREDICTIONS_WITH_SUPPORT_PATH,
    index=False,
    compression="gzip",
)

pathogen_error_summary.to_csv(
    PATHOGEN_ERROR_SUMMARY_PATH,
    index=False,
)

error_linkage_summary = pd.DataFrame(
    [
        {
            "metric": "Pathogens with Model C errors",
            "value": len(pathogen_error_summary),
        },
        {
            "metric": "Model C MIC predictions",
            "value": len(model_c_predictions),
        },
        {
            "metric": "Model 3B baseline MIC predictions",
            "value": int(
                (
                    predictions_with_support[
                        "model"
                    ]
                    == "Model 3B baseline"
                ).sum()
            ),
        },
        {
            "metric": "Missing similarity values",
            "value": int(
                predictions_with_support[
                    "nearest_training_similarity"
                ].isna().sum()
            ),
        },
        {
            "metric": "Linkage validation status",
            "value": "passed",
        },
    ]
)

display(error_linkage_summary)
display(pathogen_error_summary.head())

print(f"Saved: {PREDICTIONS_WITH_SUPPORT_PATH}")
print(f"Saved: {PATHOGEN_ERROR_SUMMARY_PATH}")

print(
    "\nTransition: Cell 27.7 will test the relationship "
    "between nearest-training similarity and pathogen-level error."
)


In [ ]:
# =============================================================================
# Cell 27.7
# =============================================================================

#@title Cell 27.7 - Test the similarity-error relationship
# This cell calculates Spearman correlations between nearest-training
# similarity and pathogen-level MAE and RMSE. Bootstrap intervals quantify
# uncertainty without assuming that the relationship is linear.


def bootstrap_spearman_interval(
    similarity_values,
    error_values,
    replicates,
    random_seed,
):
    similarity_values = np.asarray(
        similarity_values,
        dtype=np.float64,
    )

    error_values = np.asarray(
        error_values,
        dtype=np.float64,
    )

    random_generator = np.random.default_rng(
        random_seed
    )

    bootstrap_correlations = np.empty(
        replicates,
        dtype=np.float64,
    )

    number_of_pathogens = len(
        similarity_values
    )

    for replicate_index in range(
        replicates
    ):
        sampled_indices = random_generator.integers(
            0,
            number_of_pathogens,
            size=number_of_pathogens,
        )

        bootstrap_correlations[
            replicate_index
        ] = spearmanr(
            similarity_values[
                sampled_indices
            ],
            error_values[
                sampled_indices
            ],
        ).statistic

    finite_correlations = bootstrap_correlations[
        np.isfinite(
            bootstrap_correlations
        )
    ]

    if len(finite_correlations) < int(
        0.95 * replicates
    ):
        raise ValueError(
            "Too many invalid bootstrap correlations were produced."
        )

    lower_limit, upper_limit = np.percentile(
        finite_correlations,
        [
            2.5,
            97.5,
        ],
    )

    return (
        float(lower_limit),
        float(upper_limit),
    )


nearest_similarity_values = pathogen_error_summary[
    "nearest_training_similarity"
].to_numpy(dtype=np.float64)

relationship_rows = []

for error_number, (
    error_column,
    error_label,
) in enumerate(
    [
        (
            "pathogen_mae",
            "Pathogen-level MAE",
        ),
        (
            "pathogen_rmse",
            "Pathogen-level RMSE",
        ),
    ],
    start=1,
):
    error_values = pathogen_error_summary[
        error_column
    ].to_numpy(dtype=np.float64)

    correlation_result = spearmanr(
        nearest_similarity_values,
        error_values,
    )

    confidence_lower, confidence_upper = (
        bootstrap_spearman_interval(
            nearest_similarity_values,
            error_values,
            BOOTSTRAP_REPLICATES,
            RANDOM_SEED + error_number,
        )
    )

    relationship_rows.append(
        {
            "error_measure": error_label,
            "pathogens": len(
                pathogen_error_summary
            ),
            "spearman_correlation": float(
                correlation_result.statistic
            ),
            "p_value": float(
                correlation_result.pvalue
            ),
            "bootstrap_95_percent_lower":
                confidence_lower,
            "bootstrap_95_percent_upper":
                confidence_upper,
            "bootstrap_replicates":
                BOOTSTRAP_REPLICATES,
        }
    )


similarity_error_relationship = pd.DataFrame(
    relationship_rows
)

if not np.isfinite(
    similarity_error_relationship[
        [
            "spearman_correlation",
            "p_value",
            "bootstrap_95_percent_lower",
            "bootstrap_95_percent_upper",
        ]
    ].to_numpy(dtype=np.float64)
).all():
    raise ValueError(
        "The similarity-error relationship contains invalid values."
    )


SIMILARITY_ERROR_RELATIONSHIP_PATH = (
    NOTEBOOK27_RESULT_DIRECTORY
    / "27_similarity_error_relationship.csv"
)

similarity_error_relationship.to_csv(
    SIMILARITY_ERROR_RELATIONSHIP_PATH,
    index=False,
)

display(similarity_error_relationship)

rmse_relationship = similarity_error_relationship[
    similarity_error_relationship[
        "error_measure"
    ]
    == "Pathogen-level RMSE"
].iloc[0]

if rmse_relationship[
    "bootstrap_95_percent_upper"
] < 0:
    print(
        "Lower nearest-training similarity was associated "
        "with higher pathogen-level RMSE."
    )
elif rmse_relationship[
    "spearman_correlation"
] < 0:
    print(
        "The estimated relationship was in the expected "
        "direction, but its bootstrap interval included zero."
    )
else:
    print(
        "The held-out results did not show higher RMSE at "
        "lower nearest-training similarity."
    )

print(f"Saved: {SIMILARITY_ERROR_RELATIONSHIP_PATH}")

print(
    "\nTransition: Cell 27.8 will summarise performance "
    "across similarity groups and define the empirical support boundary."
)


In [ ]:
# =============================================================================
# Cell 27.8
# =============================================================================

#@title Cell 27.8 - Summarise similarity groups and define the support boundary
# This cell divides pathogens into ten groups by nearest-training similarity,
# compares Model C and Model 3B within those groups and defines the empirical
# boundary as the lowest similarity evaluated during pathogen-out validation.


def safe_pearson_correlation(
    observed_values,
    predicted_values,
):
    observed_values = np.asarray(
        observed_values,
        dtype=np.float64,
    )

    predicted_values = np.asarray(
        predicted_values,
        dtype=np.float64,
    )

    if (
        len(observed_values) < 2
        or np.std(observed_values) == 0
        or np.std(predicted_values) == 0
    ):
        return np.nan

    return float(
        np.corrcoef(
            observed_values,
            predicted_values,
        )[0, 1]
    )


pathogen_error_summary[
    "similarity_group"
] = (
    pd.qcut(
        pathogen_error_summary[
            "nearest_training_similarity"
        ],
        q=10,
        labels=False,
        duplicates="drop",
    )
    + 1
).astype(int)

similarity_group_lookup = pathogen_error_summary[
    [
        "biosample",
        "similarity_group",
    ]
]

grouped_predictions = predictions_with_support.merge(
    similarity_group_lookup,
    on="biosample",
    how="left",
    validate="many_to_one",
)

if grouped_predictions[
    "similarity_group"
].isna().any():
    raise ValueError(
        "At least one held-out prediction has no similarity group."
    )


similarity_group_rows = []

for (
    model_name,
    similarity_group,
), group_table in grouped_predictions.groupby(
    [
        "model",
        "similarity_group",
    ],
    sort=True,
):
    observed_values = group_table[
        "observed_log2_mic"
    ].to_numpy(dtype=np.float64)

    predicted_values = group_table[
        "predicted_log2_mic"
    ].to_numpy(dtype=np.float64)

    similarity_group_rows.append(
        {
            "model": model_name,
            "similarity_group": int(
                similarity_group
            ),
            "group_interpretation": (
                "lowest similarity"
                if int(similarity_group) == 1
                else "highest similarity"
                if int(similarity_group)
                == int(
                    pathogen_error_summary[
                        "similarity_group"
                    ].max()
                )
                else "intermediate similarity"
            ),
            "pathogens": group_table[
                "biosample"
            ].nunique(),
            "observations": len(group_table),
            "minimum_nearest_similarity": float(
                group_table[
                    "nearest_training_similarity"
                ].min()
            ),
            "maximum_nearest_similarity": float(
                group_table[
                    "nearest_training_similarity"
                ].max()
            ),
            "mean_nearest_similarity": float(
                group_table[
                    "nearest_training_similarity"
                ].mean()
            ),
            "mae": float(
                group_table[
                    "absolute_error"
                ].mean()
            ),
            "rmse": float(
                np.sqrt(
                    group_table[
                        "squared_error"
                    ].mean()
                )
            ),
            "pearson_r": safe_pearson_correlation(
                observed_values,
                predicted_values,
            ),
        }
    )


similarity_group_performance = pd.DataFrame(
    similarity_group_rows
)


percentile_values = np.percentile(
    nearest_similarity[
        "nearest_training_similarity"
    ].to_numpy(dtype=np.float64),
    SIMILARITY_PERCENTILES,
)

similarity_distribution = pd.DataFrame(
    {
        "percentile": SIMILARITY_PERCENTILES,
        "nearest_training_similarity":
            percentile_values,
    }
)


EMPIRICAL_SUPPORT_BOUNDARY = float(
    nearest_similarity[
        "nearest_training_similarity"
    ].min()
)

SUPPORT_BOUNDARY_SOURCE_PATHOGENS = int(
    np.isclose(
        nearest_similarity[
            "nearest_training_similarity"
        ].to_numpy(dtype=np.float64),
        EMPIRICAL_SUPPORT_BOUNDARY,
        rtol=0.0,
        atol=1e-12,
    ).sum()
)

prediction_support_rule = pd.DataFrame(
    [
        {
            "support_status": "inside evaluated similarity range",
            "rule": (
                "nearest Model C training-pathogen similarity "
                f">= {EMPIRICAL_SUPPORT_BOUNDARY:.10f}"
            ),
            "interpretation": (
                "The pathogen is within the nearest-similarity "
                "range evaluated by nested pathogen-out validation."
            ),
        },
        {
            "support_status": "outside evaluated similarity range",
            "rule": (
                "nearest Model C training-pathogen similarity "
                f"< {EMPIRICAL_SUPPORT_BOUNDARY:.10f}"
            ),
            "interpretation": (
                "The pathogen is less similar than every pathogen "
                "evaluated by nested pathogen-out validation; do "
                "not report the MIC prediction as supported."
            ),
        },
    ]
)


SIMILARITY_GROUP_PERFORMANCE_PATH = (
    NOTEBOOK27_RESULT_DIRECTORY
    / "27_similarity_group_performance.csv"
)

SIMILARITY_DISTRIBUTION_PATH = (
    NOTEBOOK27_RESULT_DIRECTORY
    / "27_nearest_similarity_distribution.csv"
)

PREDICTION_SUPPORT_RULE_PATH = (
    NOTEBOOK27_RESULT_DIRECTORY
    / "27_prediction_support_rule.csv"
)

similarity_group_performance.to_csv(
    SIMILARITY_GROUP_PERFORMANCE_PATH,
    index=False,
)

similarity_distribution.to_csv(
    SIMILARITY_DISTRIBUTION_PATH,
    index=False,
)

prediction_support_rule.to_csv(
    PREDICTION_SUPPORT_RULE_PATH,
    index=False,
)

display(
    similarity_group_performance[
        similarity_group_performance[
            "model"
        ]
        == "Model C"
    ]
)

display(similarity_distribution)
display(prediction_support_rule)

print(
    f"Empirical support boundary: "
    f"{EMPIRICAL_SUPPORT_BOUNDARY:.10f}"
)

print(
    "This boundary describes the similarity range evaluated "
    "by Notebook 26. It is not a clinical breakpoint."
)

print(f"Saved: {SIMILARITY_GROUP_PERFORMANCE_PATH}")
print(f"Saved: {SIMILARITY_DISTRIBUTION_PATH}")
print(f"Saved: {PREDICTION_SUPPORT_RULE_PATH}")

print(
    "\nTransition: Cell 27.9 will validate the complete "
    "support analysis and save its configuration."
)


In [ ]:
# =============================================================================
# Cell 27.9
# =============================================================================

#@title Cell 27.9 - Validate and save the prediction-support configuration
# This cell verifies cohort coverage, fold separation, numerical ranges and
# output completeness, then saves the support boundary and its interpretation.

required_result_paths = [
    NEAREST_SIMILARITY_PATH,
    PREDICTIONS_WITH_SUPPORT_PATH,
    PATHOGEN_ERROR_SUMMARY_PATH,
    SIMILARITY_ERROR_RELATIONSHIP_PATH,
    SIMILARITY_GROUP_PERFORMANCE_PATH,
    SIMILARITY_DISTRIBUTION_PATH,
    PREDICTION_SUPPORT_RULE_PATH,
]

missing_result_paths = [
    result_path
    for result_path in required_result_paths
    if not result_path.exists()
]

if missing_result_paths:
    raise FileNotFoundError(
        "Notebook 27 result files are missing: "
        f"{missing_result_paths}"
    )

training_fold_leakage = int(
    (
        nearest_similarity[
            "outer_fold"
        ]
        == nearest_similarity[
            "nearest_training_outer_fold"
        ]
    ).sum()
)

similarity_values = nearest_similarity[
    "nearest_training_similarity"
].to_numpy(dtype=np.float64)

all_checks = {
    "one_similarity_per_pathogen": (
        len(nearest_similarity)
        == EXPECTED_MODEL_C_PATHOGENS
        and not nearest_similarity[
            "biosample"
        ].duplicated().any()
    ),
    "one_error_summary_per_pathogen": (
        len(pathogen_error_summary)
        == EXPECTED_MODEL_C_PATHOGENS
        and not pathogen_error_summary[
            "biosample"
        ].duplicated().any()
    ),
    "all_predictions_linked": (
        len(predictions_with_support)
        == 2 * EXPECTED_MODEL_C_INTERACTIONS
        and not predictions_with_support[
            "nearest_training_similarity"
        ].isna().any()
    ),
    "no_training_fold_leakage": (
        training_fold_leakage == 0
    ),
    "all_similarity_values_finite": (
        np.isfinite(
            similarity_values
        ).all()
    ),
    "similarity_values_in_zero_one_range": (
        similarity_values.min() >= -1e-6
        and similarity_values.max()
        <= 1.0 + 1e-6
    ),
    "boundary_equals_observed_minimum": (
        math.isclose(
            EMPIRICAL_SUPPORT_BOUNDARY,
            float(similarity_values.min()),
            rel_tol=0.0,
            abs_tol=1e-12,
        )
    ),
    "all_five_folds_represented": (
        sorted(
            nearest_similarity[
                "outer_fold"
            ].astype(int).unique()
        )
        == list(
            range(
                1,
                EXPECTED_OUTER_FOLDS + 1,
            )
        )
    ),
}

if not all(all_checks.values()):
    failed_checks = [
        check_name
        for check_name, passed
        in all_checks.items()
        if not passed
    ]

    raise ValueError(
        "Notebook 27 validation failed: "
        f"{failed_checks}"
    )


rmse_relationship_record = (
    similarity_error_relationship[
        similarity_error_relationship[
            "error_measure"
        ]
        == "Pathogen-level RMSE"
    ].iloc[0]
)

if rmse_relationship_record[
    "bootstrap_95_percent_upper"
] < 0:
    RELATIONSHIP_INTERPRETATION = (
        "Lower nearest-training similarity was associated "
        "with higher pathogen-level RMSE."
    )
elif rmse_relationship_record[
    "spearman_correlation"
] < 0:
    RELATIONSHIP_INTERPRETATION = (
        "The estimated relationship was in the expected "
        "direction, but the bootstrap interval included zero."
    )
else:
    RELATIONSHIP_INTERPRETATION = (
        "The held-out data did not show higher RMSE at "
        "lower nearest-training similarity."
    )


SUPPORT_CONFIGURATION_PATH = (
    NOTEBOOK27_RESULT_DIRECTORY
    / "27_model_c_prediction_support_configuration.json"
)

VALIDATION_SUMMARY_PATH = (
    NOTEBOOK27_RESULT_DIRECTORY
    / "27_validation_summary.csv"
)

support_configuration = {
    "notebook": 27,
    "model": "Model C",
    "model_c_pathogens": EXPECTED_MODEL_C_PATHOGENS,
    "observed_interactions": EXPECTED_MODEL_C_INTERACTIONS,
    "outer_validation_folds": EXPECTED_OUTER_FOLDS,
    "selected_sequence_kernel": SELECTED_SEQUENCE_KERNEL,
    "selected_rho": SELECTED_RHO,
    "selected_pathogen_dimension":
        SELECTED_PATHOGEN_DIMENSION,
    "selected_ridge_alpha": SELECTED_RIDGE_ALPHA,
    "similarity_measure": (
        "Maximum final Model C kernel similarity to a "
        "pathogen in the corresponding outer training group"
    ),
    "empirical_support_boundary":
        EMPIRICAL_SUPPORT_BOUNDARY,
    "boundary_definition": (
        "Minimum nearest-training similarity observed among "
        "all 9,058 pathogens during nested pathogen-out validation"
    ),
    "boundary_source_pathogens":
        SUPPORT_BOUNDARY_SOURCE_PATHOGENS,
    "inside_evaluated_range_rule": (
        "nearest_training_similarity >= "
        f"{EMPIRICAL_SUPPORT_BOUNDARY:.10f}"
    ),
    "outside_evaluated_range_rule": (
        "nearest_training_similarity < "
        f"{EMPIRICAL_SUPPORT_BOUNDARY:.10f}"
    ),
    "rmse_similarity_spearman_correlation": float(
        rmse_relationship_record[
            "spearman_correlation"
        ]
    ),
    "rmse_similarity_bootstrap_95_percent_interval": [
        float(
            rmse_relationship_record[
                "bootstrap_95_percent_lower"
            ]
        ),
        float(
            rmse_relationship_record[
                "bootstrap_95_percent_upper"
            ]
        ),
    ],
    "relationship_interpretation":
        RELATIONSHIP_INTERPRETATION,
    "boundary_interpretation": (
        "This is an empirical model-support boundary, not a "
        "clinical breakpoint or guarantee of prediction accuracy."
    ),
    "validation_status": "passed",
}

temporary_configuration_path = Path(
    str(SUPPORT_CONFIGURATION_PATH)
    + ".partial"
)

with open(
    temporary_configuration_path,
    "w",
    encoding="utf-8",
) as configuration_file:
    json.dump(
        support_configuration,
        configuration_file,
        indent=2,
    )

temporary_configuration_path.replace(
    SUPPORT_CONFIGURATION_PATH
)


validation_summary = pd.DataFrame(
    [
        {
            "metric": "Model C pathogens",
            "value": EXPECTED_MODEL_C_PATHOGENS,
        },
        {
            "metric": "Held-out Model C MIC predictions",
            "value": EXPECTED_MODEL_C_INTERACTIONS,
        },
        {
            "metric": "Outer folds",
            "value": EXPECTED_OUTER_FOLDS,
        },
        {
            "metric": "Training-fold leakage",
            "value": training_fold_leakage,
        },
        {
            "metric": "Minimum evaluated similarity",
            "value": EMPIRICAL_SUPPORT_BOUNDARY,
        },
        {
            "metric": "Maximum evaluated similarity",
            "value": float(
                similarity_values.max()
            ),
        },
        {
            "metric": "RMSE-similarity Spearman correlation",
            "value": float(
                rmse_relationship_record[
                    "spearman_correlation"
                ]
            ),
        },
        {
            "metric": "Notebook 27 validation status",
            "value": "passed",
        },
    ]
)

validation_summary.to_csv(
    VALIDATION_SUMMARY_PATH,
    index=False,
)

display(validation_summary)

print(f"Saved: {SUPPORT_CONFIGURATION_PATH}")
print(f"Saved: {VALIDATION_SUMMARY_PATH}")

print(
    "\nTransition: Cell 27.10 will package the validated "
    "Notebook 27 outputs and report the final status."
)


In [ ]:
# =============================================================================
# Cell 27.10
# =============================================================================

#@title Cell 27.10 - Package and report the final Notebook 27 outputs
# This cell records checksums, creates one validated ZIP archive and reports
# the final pathogen-similarity support result.

FINAL_OUTPUT_ARCHIVE_PATH = (
    NOTEBOOK27_DIRECTORY
    / "27_model_c_pathogen_similarity_support_outputs.zip"
)

OUTPUT_MANIFEST_PATH = (
    NOTEBOOK27_RESULT_DIRECTORY
    / "27_output_manifest.json"
)

files_to_package = [
    NEAREST_SIMILARITY_PATH,
    PREDICTIONS_WITH_SUPPORT_PATH,
    PATHOGEN_ERROR_SUMMARY_PATH,
    SIMILARITY_ERROR_RELATIONSHIP_PATH,
    SIMILARITY_GROUP_PERFORMANCE_PATH,
    SIMILARITY_DISTRIBUTION_PATH,
    PREDICTION_SUPPORT_RULE_PATH,
    SUPPORT_CONFIGURATION_PATH,
    VALIDATION_SUMMARY_PATH,
]

missing_output_files = [
    file_path
    for file_path in files_to_package
    if not file_path.exists()
]

if missing_output_files:
    raise FileNotFoundError(
        "Notebook 27 output files are missing: "
        f"{missing_output_files}"
    )

output_manifest = {
    "notebook": 27,
    "model": "Model C",
    "model_c_pathogens": EXPECTED_MODEL_C_PATHOGENS,
    "held_out_model_c_predictions":
        EXPECTED_MODEL_C_INTERACTIONS,
    "outer_validation_folds": EXPECTED_OUTER_FOLDS,
    "selected_sequence_kernel":
        SELECTED_SEQUENCE_KERNEL,
    "selected_rho": SELECTED_RHO,
    "empirical_support_boundary":
        EMPIRICAL_SUPPORT_BOUNDARY,
    "rmse_similarity_spearman_correlation": float(
        rmse_relationship_record[
            "spearman_correlation"
        ]
    ),
    "relationship_interpretation":
        RELATIONSHIP_INTERPRETATION,
    "files": [
        {
            "file_name": file_path.name,
            "size_bytes": file_path.stat().st_size,
            "sha256": file_sha256(file_path),
        }
        for file_path in files_to_package
    ],
    "validation_status": "passed",
}

temporary_manifest_path = Path(
    str(OUTPUT_MANIFEST_PATH)
    + ".partial"
)

with open(
    temporary_manifest_path,
    "w",
    encoding="utf-8",
) as manifest_file:
    json.dump(
        output_manifest,
        manifest_file,
        indent=2,
    )

temporary_manifest_path.replace(
    OUTPUT_MANIFEST_PATH
)

files_to_package.append(
    OUTPUT_MANIFEST_PATH
)

local_archive_path = (
    WORK_DIRECTORY
    / FINAL_OUTPUT_ARCHIVE_PATH.name
)

local_archive_path.unlink(
    missing_ok=True
)

with zipfile.ZipFile(
    local_archive_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=3,
) as archive:
    for file_path in files_to_package:
        archive.write(
            file_path,
            arcname=file_path.name,
        )

with zipfile.ZipFile(
    local_archive_path,
    "r",
) as archive:
    damaged_member = archive.testzip()

    if damaged_member is not None:
        raise ValueError(
            "The local Notebook 27 archive contains a "
            f"damaged file: {damaged_member}"
        )

    archived_members = set(
        archive.namelist()
    )

expected_members = {
    file_path.name
    for file_path in files_to_package
}

if archived_members != expected_members:
    raise ValueError(
        "The Notebook 27 archive member list is incomplete."
    )

local_archive_sha256 = file_sha256(
    local_archive_path
)

partial_archive_path = Path(
    str(FINAL_OUTPUT_ARCHIVE_PATH)
    + ".partial"
)

partial_archive_path.unlink(
    missing_ok=True
)

shutil.copy2(
    local_archive_path,
    partial_archive_path,
)

if file_sha256(
    partial_archive_path
) != local_archive_sha256:
    raise IOError(
        "The copied Notebook 27 archive does not match "
        "the locally validated archive."
    )

partial_archive_path.replace(
    FINAL_OUTPUT_ARCHIVE_PATH
)

with zipfile.ZipFile(
    FINAL_OUTPUT_ARCHIVE_PATH,
    "r",
) as archive:
    if archive.testzip() is not None:
        raise ValueError(
            "The saved Notebook 27 archive failed validation."
        )


final_summary = pd.DataFrame(
    [
        {
            "metric": "Model C pathogens",
            "value": EXPECTED_MODEL_C_PATHOGENS,
        },
        {
            "metric": "Held-out Model C MIC predictions",
            "value": EXPECTED_MODEL_C_INTERACTIONS,
        },
        {
            "metric": "Selected sequence kernel",
            "value": SELECTED_SEQUENCE_KERNEL,
        },
        {
            "metric": "Selected rho",
            "value": SELECTED_RHO,
        },
        {
            "metric": "Minimum evaluated nearest similarity",
            "value": EMPIRICAL_SUPPORT_BOUNDARY,
        },
        {
            "metric": "RMSE-similarity Spearman correlation",
            "value": float(
                rmse_relationship_record[
                    "spearman_correlation"
                ]
            ),
        },
        {
            "metric": "Relationship interpretation",
            "value": RELATIONSHIP_INTERPRETATION,
        },
        {
            "metric": "Final archive size (MB)",
            "value": round(
                FINAL_OUTPUT_ARCHIVE_PATH.stat().st_size
                / (1024 ** 2),
                3,
            ),
        },
        {
            "metric": "Notebook 27 validation status",
            "value": "passed",
        },
    ]
)

display(final_summary)

print(f"Saved: {OUTPUT_MANIFEST_PATH}")
print(f"Saved: {FINAL_OUTPUT_ARCHIVE_PATH}")

print(
    "\nNotebook 27 completed successfully. The empirical "
    "Model C prediction-support boundary is ready for the "
    "later MIC-matrix and new-pathogen prediction notebooks."
)
